# Parte 3 — Ecuaciones elípticas
## 3.6 Series de Fourier y solución de Dirichlet en un dominio cuadrado
### 3.6.01 Separación de variables, compatibilidad en las esquinas y solución espectral

Este notebook cubre el apartado **3.6** del temario oficial:

> Series de Fourier y solución al problema de Dirichlet en un dominio cuadrado.

## Relación con las notas manuscritas

Las páginas manuscritas disponibles terminan en función de Green y no contienen
un desarrollo separado del problema rectangular. Por tanto, esta unidad se marca
como **Complemento para cerrar el temario oficial**.

Se preservan las convenciones ya fijadas:

$$
\Delta u=u_{xx}+u_{yy},
$$

y el problema se estudia primero en

$$
R=(0,a)\times(0,b).
$$

## Contenido

1. separación de variables;
2. familia $\sinh$-$\sin$;
3. superposición de los cuatro lados;
4. compatibilidad en las esquinas;
5. convergencia y regularidad interior;
6. familia $\cosh$-$\cos$ en un rectángulo centrado;
7. unicidad por el principio del máximo;
8. solucionador espectral GPU/CPU.

## Estado de las fuentes

Las capturas antes enlazadas del temario y del Examen General 2025-2 no están incluidas en el repositorio. El desarrollo que sigue es autocontenido. Para cotejar el enunciado oficial debe consultarse el PDF original fuera de este repositorio; no se sustituye aquí por una imagen inventada.


# Simulaciones y visualizaciones

Las celdas se ejecutan directamente. No existe una bandera `VIDEO=True`.

Se incluyen:

1. solucionador espectral para datos en los cuatro lados;
2. animación de convergencia de las sumas parciales;
3. estudio de incompatibilidad en las esquinas;
4. familia simétrica $\cosh$-$\cos$;
5. diagnóstico del residual discreto y de los valores de frontera.

CuPy/CUDA se utiliza automáticamente cuando hay un dispositivo disponible.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        raise RuntimeError("No se encontró un dispositivo CUDA.")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    if GPU_AVAILABLE:
        return cp.asnumpy(array)
    return np.asarray(array)


def save_and_display_animation(
    animation,
    stem,
    fps=60,
    dpi=150,
    bitrate=10000,
):
    """Guarda y muestra automáticamente una animación."""
    if shutil.which("ffmpeg"):
        output = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(output, writer=writer, dpi=dpi)
        display(Video(str(output), embed=True))
    else:
        output = ANIM_DIR / f"{stem}.gif"
        writer = PillowWriter(fps=min(fps, 35))
        animation.save(
            output,
            writer=writer,
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(output)))

    print("Animación guardada en:", output.resolve())
    return output


def stable_sinh_ratio(
    coordinate,
    length,
    wave_numbers,
):
    """
    Calcula sinh(k*coordinate)/sinh(k*length) sin desbordamiento.

    coordinate puede ser un vector columna y wave_numbers un vector fila.
    """
    coordinate = xp.asarray(coordinate)
    wave_numbers = xp.asarray(wave_numbers)

    numerator_factor = -xp.expm1(
        -2.0 * wave_numbers * coordinate
    )
    denominator_factor = -xp.expm1(
        -2.0 * wave_numbers * length
    )

    return (
        xp.exp(
            wave_numbers * (coordinate - length)
        )
        * numerator_factor
        / denominator_factor
    )


def sine_coefficients(
    values,
    grid,
    length,
    n_modes,
):
    """Coeficientes seno por cuadratura trapezoidal."""
    values = xp.asarray(values)
    grid = xp.asarray(grid)

    mode_numbers = xp.arange(
        1,
        n_modes + 1,
        dtype=xp.float64,
    )
    basis = xp.sin(
        math.pi
        * mode_numbers[:, None]
        * grid[None, :]
        / length
    )

    integrand = basis * values[None, :]

    if hasattr(xp, "trapezoid"):
        integrals = xp.trapezoid(
            integrand,
            grid,
            axis=1,
        )
    else:
        integrals = xp.trapz(
            integrand,
            grid,
            axis=1,
        )

    return 2.0 / length * integrals


def rectangle_dirichlet_solution(
    x,
    y,
    a,
    b,
    bottom_values,
    top_values,
    left_values,
    right_values,
    n_modes,
):
    """
    Solución por superposición de cuatro problemas con tres lados homogéneos.

    x: malla unidimensional de [0,a]
    y: malla unidimensional de [0,b]
    bottom_values, top_values: evaluados sobre x
    left_values, right_values: evaluados sobre y
    """
    x = xp.asarray(x)
    y = xp.asarray(y)

    mode_x = xp.arange(
        1,
        n_modes + 1,
        dtype=xp.float64,
    )
    kx = math.pi * mode_x / a
    sin_x = xp.sin(kx[:, None] * x[None, :])

    bottom_coeff = sine_coefficients(
        bottom_values,
        x,
        a,
        n_modes,
    )
    top_coeff = sine_coefficients(
        top_values,
        x,
        a,
        n_modes,
    )

    ratio_bottom = stable_sinh_ratio(
        b - y[:, None],
        b,
        kx[None, :],
    )
    ratio_top = stable_sinh_ratio(
        y[:, None],
        b,
        kx[None, :],
    )

    horizontal_part = (
        (ratio_bottom * bottom_coeff[None, :])
        @ sin_x
        + (ratio_top * top_coeff[None, :])
        @ sin_x
    )

    mode_y = xp.arange(
        1,
        n_modes + 1,
        dtype=xp.float64,
    )
    ky = math.pi * mode_y / b
    sin_y = xp.sin(ky[:, None] * y[None, :])

    left_coeff = sine_coefficients(
        left_values,
        y,
        b,
        n_modes,
    )
    right_coeff = sine_coefficients(
        right_values,
        y,
        b,
        n_modes,
    )

    ratio_left = stable_sinh_ratio(
        a - x[:, None],
        a,
        ky[None, :],
    )
    ratio_right = stable_sinh_ratio(
        x[:, None],
        a,
        ky[None, :],
    )

    vertical_part = (
        sin_y.T
        @ (left_coeff[:, None] * ratio_left.T)
        + sin_y.T
        @ (right_coeff[:, None] * ratio_right.T)
    )

    return horizontal_part + vertical_part

## Simulación 3.6.A — Solución espectral con datos compatibles en los cuatro lados

Se usa el cuadrado unitario y datos que se anulan en las esquinas:

$$
g_0(x)=0.25\sin(\pi x),
$$

$$
g_1(x)=\sin(2\pi x)+0.3\sin(5\pi x),
$$

$$
h_0(y)=0.4\sin(\pi y),
$$

$$
h_1(y)=-0.3\sin(3\pi y).
$$

La solución es la superposición de cuatro familias $\sinh$-$\sin$.

In [ ]:
a = 1.0
b = 1.0

grid_n = 360 if GPU_AVAILABLE else 190
x = xp.linspace(0.0, a, grid_n)
y = xp.linspace(0.0, b, grid_n)

bottom = 0.25 * xp.sin(math.pi * x)
top = (
    xp.sin(2.0 * math.pi * x)
    + 0.3 * xp.sin(5.0 * math.pi * x)
)
left = 0.4 * xp.sin(math.pi * y)
right = -0.3 * xp.sin(3.0 * math.pi * y)

mode_counts = np.unique(
    np.round(
        np.geomspace(1, 120, 150)
    ).astype(int)
)

frames = []

for count in mode_counts:
    solution = rectangle_dirichlet_solution(
        x,
        y,
        a,
        b,
        bottom,
        top,
        left,
        right,
        int(count),
    )
    frames.append(
        to_cpu(solution).astype(np.float32)
    )

frames = np.asarray(frames)

print("Malla:", grid_n, "x", grid_n)
print("Máximo número de modos:", int(mode_counts[-1]))

In [ ]:
vmax = float(np.max(np.abs(frames[-1])))

fig, ax = plt.subplots(figsize=(8.5, 7.0))

image = ax.imshow(
    frames[0],
    origin="lower",
    extent=[0.0, a, 0.0, b],
    interpolation="bilinear",
    vmin=-vmax,
    vmax=vmax,
)
fig.colorbar(image, ax=ax, label=r"$u_N(x,y)$")

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Convergencia de la solución espectral")

status = ax.text(
    0.02,
    0.97,
    "",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "alpha": 0.8},
)


def update_partial_sum(frame):
    image.set_data(frames[frame])
    status.set_text(
        rf"$N={mode_counts[frame]}$"
    )
    return image, status


animation = FuncAnimation(
    fig,
    update_partial_sum,
    frames=len(frames),
    interval=1000.0 / 60.0,
    blit=False,
)

fig.tight_layout()

save_and_display_animation(
    animation,
    "03.6.A_convergencia_espectral_cuadrado",
    fps=60,
    dpi=165,
    bitrate=13000,
)

plt.close(fig)

In [ ]:
# Diagnóstico discreto de armonicidad para la solución final.

u_final = frames[-1]
x_cpu = to_cpu(x)
y_cpu = to_cpu(y)

dx = float(x_cpu[1] - x_cpu[0])
dy = float(y_cpu[1] - y_cpu[0])

laplacian = (
    (
        u_final[1:-1, 2:]
        - 2.0 * u_final[1:-1, 1:-1]
        + u_final[1:-1, :-2]
    )
    / dx**2
    + (
        u_final[2:, 1:-1]
        - 2.0 * u_final[1:-1, 1:-1]
        + u_final[:-2, 1:-1]
    )
    / dy**2
)

margin = max(4, grid_n // 30)
interior_laplacian = laplacian[
    margin:-margin,
    margin:-margin,
]

print(
    "Norma sup del laplaciano discreto lejos de la frontera:",
    float(np.max(np.abs(interior_laplacian))),
)

fig, ax = plt.subplots(figsize=(8.5, 7))
image = ax.imshow(
    u_final,
    origin="lower",
    extent=[0.0, a, 0.0, b],
    interpolation="bilinear",
)
fig.colorbar(image, ax=ax, label=r"$u(x,y)$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Solución de Dirichlet por superposición")
fig.tight_layout()

path = FIG_DIR / "03.6.A_solucion_cuatro_lados.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

## Simulación 3.6.B — Compatibilidad e incompatibilidad en las esquinas

Para una solución continua en $\overline R$, los cuatro datos deben coincidir en
cada esquina:

$$
g_0(0)=h_0(0),
\qquad
g_0(a)=h_1(0),
$$

$$
g_1(0)=h_0(b),
\qquad
g_1(a)=h_1(b).
$$

Se compara:

- un dato superior compatible, $g_1(x)=4x(1-x)$;
- un dato superior incompatible, $g_1(x)=1$;

manteniendo cero en los otros tres lados.

In [ ]:
# Problema con tres lados cero.
n_corner = 360 if GPU_AVAILABLE else 210
x_corner = xp.linspace(0.0, 1.0, n_corner)
y_corner = xp.linspace(0.0, 1.0, n_corner)

zero_x = xp.zeros_like(x_corner)
zero_y = xp.zeros_like(y_corner)

top_compatible = 4.0 * x_corner * (1.0 - x_corner)
top_incompatible = xp.ones_like(x_corner)

u_compatible = rectangle_dirichlet_solution(
    x_corner,
    y_corner,
    1.0,
    1.0,
    zero_x,
    top_compatible,
    zero_y,
    zero_y,
    180,
)

u_incompatible = rectangle_dirichlet_solution(
    x_corner,
    y_corner,
    1.0,
    1.0,
    zero_x,
    top_incompatible,
    zero_y,
    zero_y,
    180,
)

u_compatible = to_cpu(u_compatible)
u_incompatible = to_cpu(u_incompatible)
x_corner_cpu = to_cpu(x_corner)
y_corner_cpu = to_cpu(y_corner)

# Perfiles paralelos a la frontera superior.
levels = [0.75, 0.90, 0.97, 0.992]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    x_corner_cpu,
    np.ones_like(x_corner_cpu),
    linestyle="--",
    label="dato superior incompatible",
)

for level in levels:
    index = int(
        np.argmin(np.abs(y_corner_cpu - level))
    )
    ax.plot(
        x_corner_cpu,
        u_incompatible[index, :],
        label=rf"$y={y_corner_cpu[index]:.3f}$",
    )

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$u(x,y)$")
ax.set_title("Aproximación al dato incompatible cerca de las esquinas")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

path = FIG_DIR / "03.6.B_incompatibilidad_esquinas.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())

In [ ]:
# Comparación visual de ambos problemas.

for data, title, filename in [
    (
        u_compatible,
        "Dato compatible en las esquinas",
        "03.6.B_solucion_compatible.png",
    ),
    (
        u_incompatible,
        "Dato incompatible en las esquinas",
        "03.6.B_solucion_incompatible.png",
    ),
]:
    fig, ax = plt.subplots(figsize=(8, 6.8))
    image = ax.imshow(
        data,
        origin="lower",
        extent=[0.0, 1.0, 0.0, 1.0],
        interpolation="bilinear",
    )
    fig.colorbar(image, ax=ax)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_title(title)
    fig.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=220)
    plt.show()
    plt.close(fig)
    print(path.resolve())

## Simulación 3.6.C — Familia $\cosh$-$\cos$ en un rectángulo centrado

Consideramos

$$
R_c=(-a,a)\times(-b,b)
$$

con dato par e igual en las fronteras superior e inferior y dato cero en los
lados verticales.

Las frecuencias

$$
\mu_k=\frac{(2k+1)\pi}{2a}
$$

satisfacen

$$
\cos(\mu_k a)=0.
$$

Por tanto, la familia

$$
\frac{\cosh(\mu_k y)}{\cosh(\mu_k b)}
\cos(\mu_k x)
$$

es armónica, se anula en $x=\pm a$ y toma el mismo valor en $y=\pm b$.

In [ ]:
# ============================================================
# FAMILIA cosh-cos
# ============================================================

a_center = 1.0
b_center = 0.75

n_center = 360 if GPU_AVAILABLE else 210
xc = xp.linspace(-a_center, a_center, n_center)
yc = xp.linspace(-b_center, b_center, n_center)

boundary_even = (
    1.0 - (xc / a_center) ** 2
)

n_modes_center = 90
k_index = xp.arange(
    0,
    n_modes_center,
    dtype=xp.float64,
)
mu = (
    (2.0 * k_index + 1.0)
    * math.pi
    / (2.0 * a_center)
)

cos_basis = xp.cos(mu[:, None] * xc[None, :])

if hasattr(xp, "trapezoid"):
    coefficients = (
        1.0
        / a_center
        * xp.trapezoid(
            cos_basis * boundary_even[None, :],
            xc,
            axis=1,
        )
    )
else:
    coefficients = (
        1.0
        / a_center
        * xp.trapz(
            cos_basis * boundary_even[None, :],
            xc,
            axis=1,
        )
    )

# cociente cosh(mu*y)/cosh(mu*b) estable.
abs_y = xp.abs(yc[:, None])
cosh_ratio = (
    xp.exp(mu[None, :] * (abs_y - b_center))
    * (
        1.0
        + xp.exp(-2.0 * mu[None, :] * abs_y)
    )
    / (
        1.0
        + xp.exp(-2.0 * mu[None, :] * b_center)
    )
)

u_center = (
    cosh_ratio
    * coefficients[None, :]
) @ cos_basis

u_center_cpu = to_cpu(u_center)

fig, ax = plt.subplots(figsize=(8.5, 6.8))
image = ax.imshow(
    u_center_cpu,
    origin="lower",
    extent=[
        -a_center,
        a_center,
        -b_center,
        b_center,
    ],
    interpolation="bilinear",
)
fig.colorbar(image, ax=ax, label=r"$u(x,y)$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title(r"Familia $\cosh$-$\cos$ con simetría par")
fig.tight_layout()

path = FIG_DIR / "03.6.C_familia_cosh_cos.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

print(
    "Error de simetría y -> -y:",
    float(
        np.max(
            np.abs(
                u_center_cpu
                - u_center_cpu[::-1, :]
            )
        )
    ),
)
print(path.resolve())

# 3.6.1 Separación de variables

## **Complemento para cerrar el temario oficial**

Buscamos soluciones no triviales de

$$
\Delta u=0
$$

en el rectángulo

$$
R=(0,a)\times(0,b)
$$

de la forma

$$
u(x,y)=X(x)Y(y).
$$

Sustituyendo,

$$
X''(x)Y(y)+X(x)Y''(y)=0.
$$

Dividiendo entre $X(x)Y(y)$,

$$
\frac{X''(x)}{X(x)}
=
-\frac{Y''(y)}{Y(y)}
=
-\lambda.
$$

Se obtiene el sistema

$$
X''+\lambda X=0,
$$

$$
Y''-\lambda Y=0.
$$

## **Proposición 3.6.1 (Familia $\sinh$-$\sin$).**

Supongamos que

$$
u(0,y)=u(a,y)=u(x,0)=0.
$$

Los valores propios del problema en $x$ son

$$
\lambda_n
=
\left(\frac{n\pi}{a}\right)^2,
\qquad
n\geq1,
$$

con funciones propias

$$
X_n(x)
=
\sin\left(\frac{n\pi x}{a}\right).
$$

Imponiendo $Y_n(0)=0$ se obtiene

$$
Y_n(y)
=
\sinh\left(\frac{n\pi y}{a}\right).
$$

Por tanto,

$$
u_n(x,y)
=
\sinh\left(\frac{n\pi y}{a}\right)
\sin\left(\frac{n\pi x}{a}\right)
$$

es armónica y se anula en tres lados.

# 3.6.2 Un lado no homogéneo

## **Teorema 3.6.2 (Dirichlet con dato en el lado superior).**

Sea $f\in C([0,a])$ y supongamos

$$
f(0)=f(a)=0.
$$

El problema

$$
\begin{cases}
\Delta u=0,
& (x,y)\in(0,a)\times(0,b),\\
u(0,y)=u(a,y)=u(x,0)=0,\\
u(x,b)=f(x)
\end{cases}
$$

tiene la solución

$$
u(x,y)
=
\sum_{n=1}^{\infty}
b_n
\frac{
\sinh\left(\frac{n\pi y}{a}\right)
}{
\sinh\left(\frac{n\pi b}{a}\right)
}
\sin\left(\frac{n\pi x}{a}\right),
$$

donde

$$
b_n
=
\frac{2}{a}
\int_0^a
f(s)
\sin\left(\frac{n\pi s}{a}\right)ds.
$$

### Demostración añadida

La separación de variables produce la familia de la Proposición 3.6.1. El dato
superior exige

$$
f(x)
=
\sum_{n=1}^{\infty}
b_n
\sin\left(\frac{n\pi x}{a}\right),
$$

que es la serie seno de Fourier de $f$. La normalización por
$\sinh(n\pi b/a)$ garantiza que el término $n$ toma el valor
$b_n\sin(n\pi x/a)$ sobre $y=b$.

La convergencia y la continuidad hasta la frontera se justifican en la
Sección 3.6.5.

$\square$

## **Corolario 3.6.3 (Datos inferior, izquierdo y derecho).**

Con la misma notación:

### Dato inferior $g_0$

$$
u_{\mathrm{inf}}(x,y)
=
\sum_{n=1}^{\infty}
a_n
\frac{
\sinh\left(\frac{n\pi(b-y)}{a}\right)
}{
\sinh\left(\frac{n\pi b}{a}\right)
}
\sin\left(\frac{n\pi x}{a}\right).
$$

### Dato superior $g_1$

$$
u_{\mathrm{sup}}(x,y)
=
\sum_{n=1}^{\infty}
b_n
\frac{
\sinh\left(\frac{n\pi y}{a}\right)
}{
\sinh\left(\frac{n\pi b}{a}\right)
}
\sin\left(\frac{n\pi x}{a}\right).
$$

### Dato izquierdo $h_0$

$$
u_{\mathrm{izq}}(x,y)
=
\sum_{n=1}^{\infty}
c_n
\frac{
\sinh\left(\frac{n\pi(a-x)}{b}\right)
}{
\sinh\left(\frac{n\pi a}{b}\right)
}
\sin\left(\frac{n\pi y}{b}\right).
$$

### Dato derecho $h_1$

$$
u_{\mathrm{der}}(x,y)
=
\sum_{n=1}^{\infty}
d_n
\frac{
\sinh\left(\frac{n\pi x}{b}\right)
}{
\sinh\left(\frac{n\pi a}{b}\right)
}
\sin\left(\frac{n\pi y}{b}\right).
$$

Los coeficientes son las correspondientes series seno de Fourier.

# 3.6.3 Superposición para los cuatro lados

## **Teorema 3.6.4 (Solución por superposición).**

Sean los datos

$$
g_0,g_1\in C([0,a]),
$$

$$
h_0,h_1\in C([0,b]).
$$

Supongamos que son compatibles en las esquinas:

$$
g_0(0)=h_0(0),
\qquad
g_0(a)=h_1(0),
$$

$$
g_1(0)=h_0(b),
\qquad
g_1(a)=h_1(b).
$$

Entonces el problema

$$
\begin{cases}
\Delta u=0,
& \text{en }R,\\
u(x,0)=g_0(x),\\
u(x,b)=g_1(x),\\
u(0,y)=h_0(y),\\
u(a,y)=h_1(y)
\end{cases}
$$

puede reducirse por una función bilineal que interpola los valores de las
esquinas y por superposición de cuatro problemas con tres lados homogéneos.

### Complemento: reducción de las esquinas

Sea $p$ la función bilineal que coincide con los cuatro valores de esquina.
Entonces

$$
\Delta p=0.
$$

Al definir

$$
v=u-p,
$$

los cuatro nuevos datos se anulan en los extremos de sus respectivos intervalos.
Por tanto, cada lado puede representarse mediante una serie seno y

$$
v
=
v_{\mathrm{inf}}
+
v_{\mathrm{sup}}
+
v_{\mathrm{izq}}
+
v_{\mathrm{der}}.
$$

Finalmente,

$$
u=p+v.
$$

### **Aclaración sobre el solucionador numérico.**

Las simulaciones de este notebook usan datos que ya se anulan en las esquinas,
por lo que $p=0$. Para datos generales compatibles debe añadirse primero la
interpolación bilineal.

# 3.6.4 Compatibilidad en las esquinas

## **Proposición 3.6.5 (Condición necesaria de compatibilidad).**

Si

$$
u\in C(\overline R)
$$

satisface los cuatro datos de Dirichlet, entonces necesariamente

$$
g_0(0)=h_0(0),
\qquad
g_0(a)=h_1(0),
$$

$$
g_1(0)=h_0(b),
\qquad
g_1(a)=h_1(b).
$$

### Demostración

Por continuidad, el valor de $u$ en cada esquina debe ser independiente del lado
desde el cual se aproxima.

$\square$

## **Observación 3.6.6 (Datos incompatibles).**

Si los datos laterales son incompatibles, no existe una solución en

$$
C^2(R)\cap C(\overline R)
$$

que tome todos los valores prescritos.

Todavía puede existir una función armónica en el interior que se aproxime a cada
dato en los puntos abiertos de los lados. Los límites en la esquina dependen de
la forma de aproximación y las derivadas pueden desarrollar una singularidad.
La Simulación 3.6.B ilustra este fenómeno.

### Ejercicios — Secciones 3.6.1 a 3.6.4

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Derive la fórmula correspondiente a un dato no homogéneo únicamente en el lado
   derecho.

2. Construya explícitamente la función bilineal que interpola cuatro valores
   prescritos en las esquinas.


# 3.6.5 Convergencia, regularidad y unicidad

## **Teorema 3.6.7 (Convergencia interior de la serie).**

Sea $f\in L^2(0,a)$ y sean $b_n$ sus coeficientes seno. Para cada
$0<\delta<b$, la serie

$$
\sum_{n=1}^{\infty}
b_n
\frac{
\sinh\left(\frac{n\pi y}{a}\right)
}{
\sinh\left(\frac{n\pi b}{a}\right)
}
\sin\left(\frac{n\pi x}{a}\right)
$$

converge absoluta y uniformemente en

$$
[0,a]\times[0,b-\delta].
$$

Además, puede derivarse término a término cualquier número finito de veces sobre
compactos contenidos en $R$.

### Demostración añadida

Para $y\leq b-\delta$,

$$
0
\leq
\frac{
\sinh\left(\frac{n\pi y}{a}\right)
}{
\sinh\left(\frac{n\pi b}{a}\right)
}
\leq
C_\delta
e^{-n\pi\delta/a}.
$$

Los coeficientes $b_n$ pertenecen a $\ell^2$ por Parseval. El producto con el
factor exponencial pertenece a $\ell^1$ por Cauchy-Schwarz. Esto da convergencia
uniforme. Las derivadas introducen potencias de $n$, que siguen dominadas por el
decaimiento exponencial.

$\square$

## **Teorema 3.6.8 (Continuidad hasta la frontera).**

Supongamos

$$
f\in C([0,a]),
\qquad
f(0)=f(a)=0.
$$

La solución del Teorema 3.6.2 pertenece a

$$
C^\infty(R)\cap C(\overline R)
$$

y recupera uniformemente el dato $f$ sobre el lado superior.

### Complemento

Puede probarse usando la fórmula de Poisson del rectángulo, teoría de
identidades aproximadas o aproximación uniforme de $f$ por polinomios seno.
La serie converge uniformemente en compactos interiores; la continuidad en las
esquinas utiliza $f(0)=f(a)=0$.

## **Teorema 3.6.9 (Unicidad).**

El problema clásico de Dirichlet en el rectángulo tiene a lo más una solución.

### Demostración

Si $u_1$ y $u_2$ son soluciones, entonces $w=u_1-u_2$ es armónica y se anula
en toda la frontera. El principio del máximo aplicado a $w$ y $-w$ implica

$$
w\equiv0.
$$

$\square$

# 3.6.6 Familia $\cosh$-$\cos$

## **Proposición 3.6.10 (Familia simétrica en un rectángulo centrado).**

Sea

$$
R_c=(-a,a)\times(-b,b)
$$

y defínanse

$$
\mu_k
=
\frac{(2k+1)\pi}{2a},
\qquad
k=0,1,2,\dots.
$$

Las funciones

$$
u_k(x,y)
=
\frac{\cosh(\mu_k y)}
{\cosh(\mu_k b)}
\cos(\mu_k x)
$$

son armónicas, se anulan en $x=\pm a$ y satisfacen

$$
u_k(x,b)=u_k(x,-b)=\cos(\mu_k x).
$$

Por tanto, si el mismo dato par $f(x)$ se prescribe en $y=\pm b$ y se anula
en $x=\pm a$, la solución se expresa como

$$
u(x,y)
=
\sum_{k=0}^{\infty}
c_k
\frac{\cosh(\mu_k y)}
{\cosh(\mu_k b)}
\cos(\mu_k x),
$$

donde

$$
c_k
=
\frac{1}{a}
\int_{-a}^{a}
f(s)\cos(\mu_k s)\,ds.
$$

### **Observación 3.6.11.**

La familia $\cosh$-$\cos$ no reemplaza a la familia $\sinh$-$\sin$; corresponde
a una geometría centrada y a una simetría distinta de los datos.

### Ejercicios — Secciones 3.6.5 y 3.6.6

> **Ruta corta de estudio:** dos ejercicios representativos; este subtema se integra después en los problemas prioritarios.

1. Pruebe con detalle la estimación exponencial del cociente de senos hiperbólicos.

2. Demuestre convergencia uniforme de las derivadas sobre compactos interiores.


# Control de cobertura y estado del capítulo

## Contenido cubierto

- separación de variables en un rectángulo;
- familias $\sinh$-$\sin$;
- superposición de datos de los cuatro lados;
- reducción bilineal de valores de esquina;
- compatibilidad e incompatibilidad;
- convergencia interior;
- continuidad hasta la frontera;
- unicidad;
- familia simétrica $\cosh$-$\cos$;
- solucionador espectral GPU/CPU;
- visualizaciones de convergencia y singularidad en esquinas.

## Relación con las fuentes

Esta unidad no transcribe una sección manuscrita inexistente. Fue añadida para
cubrir explícitamente el apartado 3.6 del programa oficial y mantener el nivel
de rigor de los exámenes históricos.

## Pendiente inmediato

El siguiente notebook de la cola es

$$
\texttt{03.7.01\_Metodo\_de\_Perron.ipynb}.
$$

Se desarrollarán familias subarmónicas, envolvente de Perron, lema de elevación,
armonía de la envolvente y recuperación del dato en puntos regulares.